# SHA-3 over the state matrix

A reference model of SHA-3 per FIPS 202, written so that every step can be
inspected. The state is a 5×5×64-bit matrix held as a numpy array of shape
(y, x, z): bit number 64(5y+x)+z of the state bit string, with the
least-significant bit of every message byte coming first. Each permutation
step is its own function, so the state can be examined after any step and
compared against the hardware. At the end the whole model is checked against
the `hashlib` implementation, including messages longer than one block.

In [ ]:
import hashlib
import numpy as np

W = 64

# Round constants for the iota step (FIPS 202).
RC = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008,
]

# Rotation offsets for the rho step, RHO[x][y] (FIPS 202).
RHO = [[0, 36, 3, 41, 18],
       [1, 44, 10, 45, 2],
       [62, 6, 43, 15, 61],
       [28, 55, 25, 21, 56],
       [27, 20, 39, 8, 14]]

## Permutation steps

One round is the composition of five steps, always in the same order:

- **theta** — every bit is XOR-ed with the parity of two neighbouring
  columns, the left one and the right one shifted by one bit along z;
- **rho** — every lane (fixed x and y, all 64 bits along z) rotates by its
  fixed offset from the `RHO` table;
- **pi** — lanes trade places: position (x, y) receives the lane from
  position (x + 3y mod 5, x);
- **chi** — the only non-linear step: every bit is XOR-ed with the product
  of its inverted first and plain second neighbour within its row;
- **iota** — the round constant enters lane (0, 0), which makes the rounds
  differ from one another.

The Keccak-f[1600] permutation is 24 such rounds.

In [ ]:
def theta(A):
    C = np.bitwise_xor.reduce(A, axis=0)                # column parity, shape (x, z)
    A_new = A.copy()
    for x in range(5):
        D = C[(x - 1) % 5] ^ np.roll(C[(x + 1) % 5], 1)
        A_new[:, x, :] ^= D
    return A_new

def rho(A):
    A_new = np.empty_like(A)
    for y in range(5):
        for x in range(5):
            A_new[y, x] = np.roll(A[y, x], RHO[x][y])
    return A_new

def pi(A):
    A_new = np.empty_like(A)
    for y in range(5):
        for x in range(5):
            A_new[y, x] = A[x, (x + 3 * y) % 5]
    return A_new

def chi(A):
    A_new = np.empty_like(A)
    for x in range(5):
        A_new[:, x, :] = A[:, x, :] ^ ((1 - A[:, (x + 1) % 5, :]) & A[:, (x + 2) % 5, :])
    return A_new

def iota(A, rnd):
    A_new = A.copy()
    for z in range(W):
        A_new[0, 0, z] ^= (RC[rnd] >> z) & 1
    return A_new

def keccak_f(A):
    for rnd in range(24):
        A = iota(chi(pi(rho(theta(A)))), rnd)
    return A

## Conversions

The message unrolls into bits byte by byte, least-significant bit first.
The same order applies in the opposite direction, so the state or a digest
reads back as a hexadecimal byte string.

In [ ]:
def bytes_to_bits(data):
    bits = np.zeros(len(data) * 8, dtype=np.uint8)
    for i, b in enumerate(data):
        for j in range(8):
            bits[8 * i + j] = (b >> j) & 1
    return bits

def bits_to_bytes(bits):
    out = bytearray()
    for i in range(0, len(bits), 8):
        b = 0
        for j in range(8):
            b |= int(bits[i + j]) << j
        out.append(b)
    return bytes(out)

def show_state(A):
    return bits_to_bytes(A.reshape(-1)).hex(' ')

## The sponge

The message is padded with byte `06`, zeros up to the block boundary and
bit `80` in the final byte (when the padding fits into a single byte, that
byte is `86`). Padded like that, it is absorbed block by block of `rate_B`
bytes: the block is XOR-ed onto the first `rate_B` bytes of the state and
the state then passes through the permutation. The digest is the first
`out_bits` bits of the state after the last block.

In [ ]:
def sha3_hash(message, rate_B, out_bits):
    block = bytearray(message)
    block.append(0x06)
    if len(block) % rate_B:
        block.extend(b'\x00' * (rate_B - len(block) % rate_B))
    block[-1] |= 0x80
    A = np.zeros((5, 5, W), dtype=np.uint8)
    flat = A.reshape(-1)
    for i in range(0, len(block), rate_B):
        flat[: rate_B * 8] ^= bytes_to_bits(block[i:i + rate_B])
        A = keccak_f(A)
        flat = A.reshape(-1)
    return bits_to_bytes(flat[:out_bits])

## Check against hashlib

For two of the variants the hardware implements (SHA3-256 with a 136-byte
block and SHA3-512 with a 72-byte one) the check covers the empty message, a short
message, a message exactly at the block boundary, one byte shorter and one
byte longer than it (the padding corner cases), and a multi-block message.

In [ ]:
for name, rate_B, out_bits in [("sha3_256", 136, 256), ("sha3_512", 72, 512)]:
    lengths = [0, 3, rate_B - 1, rate_B, rate_B + 1, 3 * rate_B]
    for n in lengths:
        message = (bytes(range(256)) * 3)[:n]
        ours = sha3_hash(message, rate_B, out_bits)
        ref = getattr(hashlib, name)(message).digest()
        assert ours == ref, f"{name}, length {n}"
    print(f"{name}: matches hashlib for lengths {lengths}")